In [1]:
import argparse
import json
import tensorboard
import tensorboardX
import os
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU,MutableLinear
from nni.nas.experiment.config import NasExperimentConfig
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from nni.experiment.config import utils, ExperimentConfig
#from ops import AvgPool,DilConv,SepConv
import genotypes
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace,ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification

# Dataloader

## Data Augmentation

In [2]:
def cutout_transform(img, length: int = 16):
    h, w = img.size(1), img.size(2)
    mask = np.ones((h, w), np.float32)
    y = np.random.randint(h)
    x = np.random.randint(w)

    y1 = np.clip(y - length // 2, 0, h)
    y2 = np.clip(y + length // 2, 0, h)
    x1 = np.clip(x - length // 2, 0, w)
    x2 = np.clip(x + length // 2, 0, w)

    mask[y1: y2, x1: x2] = 0.
    mask = torch.from_numpy(mask)
    mask = mask.expand_as(img)
    img *= mask
    return img


## Data Loader Inizialization

In [3]:
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, SubsetRandomSampler
import nni
import numpy as np

def get_cifar10_dataset(train: bool = True, cutout: bool = False):
    if train:
        transform_list = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(), 
        ]
        if cutout:
            transform_list.append(cutout_transform)
        transform = transforms.Compose(transform_list)
    else:
        transform = transforms.Compose([
            transforms.ToTensor(), 
        ])

    dataset = nni.trace(CIFAR10)(root='./data', train=train, download=True, transform=transform)
    
    return dataset

batch_size = 128
train_data = get_cifar10_dataset()
test_data =get_cifar10_dataset(train=False)

train_loader = DataLoader(
    train_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True,shuffle=True
)

valid_loader = DataLoader(
    test_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True
)


Files already downloaded and verified
Files already downloaded and verified


# Classification Module

In [4]:
@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__( self,learning_rate: float = 0.001,weight_decay: float = 0.,auxiliary_loss_weight: float = 0.4,max_epochs: int = 600):
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay, export_onnx=False,num_classes=10)        
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate


    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=0.)
        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
        }

    def training_step(self, batch, batch_idx):
        """Training step, customized with auxiliary loss."""
        x, y = batch
        if self.auxiliary_loss_weight:
            y_hat, y_aux = self(x)
            loss_main = self.criterion(y_hat, y)
            loss_aux = self.criterion(y_aux, y)
            self.log('train_loss_main', loss_main)
            self.log('train_loss_aux', loss_aux)
            loss = loss_main + self.auxiliary_loss_weight * loss_aux
        else:
            y_hat = self(x)
            loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        return loss

    def on_train_epoch_start(self):
        # Set drop path probability before every epoch. This has no effect if drop path is not enabled in model.
        #self.model.set_drop_path_prob(self.model.drop_path_prob * self.current_epoch / self.max_epochs)

        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])


# Model Space

In [5]:

class CustomDARTSSpace(ModelSpace):
    def __init__(self, input_channels=3, channels=64, num_classes=10, layers=5,verbose =0, drop_path_prob = 0.1):
        super(CustomDARTSSpace, self).__init__()

        #________________________________________________________________________________________________________________________
        #Inizialization
        self.layers = nn.ModuleList()
        self.drop_path_prob = drop_path_prob
        self.verbose = verbose


        #________________________________________________________________________________________________________________________
        #Channel choices
        layer0_out = 16
        layer1_out = nni.choice('layer1_out_channels', [16,32,64])
        layer2_out= nni.choice('layer2_out_channels', [16,32,64])
        layer3_out= nni.choice('layer3_out_channels', [16,32,64])
        layer4_out = nni.choice('layer4_out_channels', [16,32,64])
        layer5_out= nni.choice('layer5_out_channels', [16,32,64])
        layer6_out= nni.choice('layer6_out_channels', [16,32,64])
        layer7_out= 22
        
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.preliminary_layer = nn.Conv2d(3, layer0_out, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(layer0_out)
        self.layer0_relu = torch.nn.ReLU(inplace=True)
        
        #________________________________________________________________________________________________________________________
        #Layer 1
        layer1 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1 ),
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer0_out, layer1_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer1_out),
                MutableReLU()
            )
        ], label='layer_1')
        self.layers.append(layer1)
        
        #________________________________________________________________________________________________________________________
        #Layer 2
        layer2 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer1_out, layer2_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer2_out),
                MutableReLU()
            )
        ], label='layer_2')
        self.layers.append(layer2)
        
        #________________________________________________________________________________________________________________________
        #Layer 3
        layer3 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer2_out, layer3_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer3_out),
                MutableReLU()
            )
        ], label='layer_3')
        self.layers.append(layer3)
                #________________________________________________________________________________________________________________________
        #Layer 4
        layer4 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer3_out, layer4_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer4_out),
                MutableReLU()
            )
        ], label='layer_4')
        self.layers.append(layer4)
                #________________________________________________________________________________________________________________________
        #Layer 5
        layer5 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer4_out, layer5_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer5_out),
                MutableReLU()
            )
        ], label='layer_5')
        self.layers.append(layer5)
                #________________________________________________________________________________________________________________________
        #Layer 6
        layer6= LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer5_out, layer6_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer6_out),
                MutableReLU()
            )
        ], label='layer_6')
        self.layers.append(layer6)
                #________________________________________________________________________________________________________________________
        #Layer 7
        layer7 = LayerChoice([
            nn.Sequential(
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            ),   
            nn.Sequential(
                MutableConv2d(layer6_out, layer7_out, kernel_size=3),
                nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                MutableBatchNorm2d(layer7_out),
                MutableReLU()
            )
        ], label='layer_7')
        self.layers.append(layer7)
        

        
        #________________________________________________________________________________________________________________________
        #Linear
        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        feature1 = nni.choice('feature1', [32, 64, 128])
        feature2 = nni.choice('feature2', [32 ,64, 128])
        feature3 = nni.choice('feature3', [32, 64])
        self.fc1 = MutableLinear(198, feature1) 
        self.fc2 = MutableLinear(feature1, feature2) 
        self.fc3 = MutableLinear(feature2, feature3)  
        self.relu = nn.ReLU()
        self.classifier = MutableLinear(feature3, num_classes)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        #Layer 0
        x = self.preliminary_layer(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        if self.verbose == 1 :
            print(f'After preliminary layer: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Layer 1 to n
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if self.verbose == 1 :
                print(f'After layer {i+1}: {x.shape}')
            if i == 1 or i == 3 or i == 6:
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
                if self.verbose == 1 :
                    print(f'After avg pooling: {x.shape}')
        
        #________________________________________________________________________________________________________________________
        #Adaprive pool
        x =  self.pool(x)
        if self.verbose == 1 :
            print(f'After adaptive pooling: {x.shape}')

        #________________________________________________________________________________________________________________________
        #Flatten
        x = torch.flatten(x, 1)
        if self.verbose == 1 :
            print(f'After flattening: {x.shape}')
        #________________________________________________________________________________________________________________________
        
        x = self.fc1(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc1: {x.shape}')
        x = self.fc2(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc2: {x.shape}')
        x = self.fc3(x)
        x= self.relu(x)
        if self.verbose == 1 :
            print(f'After fc3: {x.shape}')
        #________________________________________________________________________________________________________________________
        #Classification 
        x = self.classifier(x)

        
        if self.verbose == 1 :
            print(f'After classifier: {x.shape}')
        #self.first_iter = False
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob
        for layer in self.layers:
            if hasattr(layer, 'set_drop_path_prob'):
                layer.set_drop_path_prob(drop_path_prob)


# Evaluator

## Checkpoint callback

In [6]:
# Checkpoint 
checkpoint_callback = ModelCheckpoint(
    monitor='train_acc', 
    dirpath='./checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='max'
    
)

## Lightning Evaluator

In [7]:
from nni.nas.evaluator.pytorch import Lightning, Trainer

max_epochs = 200

evaluator = Lightning(
    DartsClassificationModule(1e-2, 0., 0., max_epochs),
    Trainer(
        accelerator="auto",
        callbacks=[checkpoint_callback],
        max_epochs=max_epochs
    ),
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


# Search

## NAS Strategy

In [8]:
def get_next_experiment_name(experiment_working_directory: str, base_name: str = "PhotonicDARTS-v"):
    dirs = os.listdir(experiment_working_directory)

    pattern = re.compile(rf'{base_name}(\d+)')

    highest_i = 0
    for d in dirs:
        match = pattern.match(d)
        if match:
            i_value = int(match.group(1))
            if i_value > highest_i:
                highest_i = i_value
    return f"{base_name}{highest_i + 1}"

In [9]:
strategy = DartsStrategy(gradient_clip_val=0.)
def search(log_dir: str, batch_size: int = 128):

    # Define model search space
    model_space = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=7, verbose=1)
    model_space.set_drop_path_prob(0.)

    # Run NAS experiment
    exp_config = NasExperimentConfig.default(model_space, evaluator, strategy)
    exp_config.experiment_working_directory = "./DartsCheckpoints"
    exp_config.experiment_name = "Darts_search"
    exp_config.trial_concurrency = 1
    experiment = NasExperiment(model_space, evaluator, strategy, config = exp_config)
    experiment.run()

    return experiment


In [10]:
experiment_results = search("./",32)

[2025-02-12 16:10:56] Config is not provided. Will try to infer.
[2025-02-12 16:10:56] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16:10:56] WARNING: `training_service` will be ignored for sequential execution engine.
[2025-02-12 16

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:653: Checkpoint directory C:\Users\senti\Documents\GitHub\PhotonicNas\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                      | Params
--------------------------------------------------------------
0 | training_module | DartsClassificationModule | 466 K 
--------------------------------------------------------------
466 K     Trainable params
0         Non-trainable params
466 K     Total params
1.865     Total estimated model params size (MB)


Epoch 0:   0%|          | 0/391 [00:00<?, ?it/s] After preliminary layer: torch.Size([128, 16, 30, 30])
After layer 1: torch.Size([128, 64, 29, 29])
After layer 2: torch.Size([128, 64, 28, 28])
After avg pooling: torch.Size([128, 64, 14, 14])
After layer 3: torch.Size([128, 64, 13, 13])
After layer 4: torch.Size([128, 64, 12, 12])
After avg pooling: torch.Size([128, 64, 6, 6])
After layer 5: torch.Size([128, 64, 5, 5])
After layer 6: torch.Size([128, 64, 4, 4])
After layer 7: torch.Size([128, 22, 3, 3])
After avg pooling: torch.Size([128, 22, 1, 1])
After adaptive pooling: torch.Size([128, 22, 3, 3])
After flattening: torch.Size([128, 198])
After fc1: torch.Size([128, 128])
After fc2: torch.Size([128, 128])
After fc3: torch.Size([128, 64])
After classifier: torch.Size([128, 10])


C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\torch\autograd\graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ..\aten\src\ATen\native\cudnn\Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


After preliminary layer: torch.Size([128, 16, 30, 30])
After layer 1: torch.Size([128, 64, 29, 29])
After layer 2: torch.Size([128, 64, 28, 28])
After avg pooling: torch.Size([128, 64, 14, 14])
After layer 3: torch.Size([128, 64, 13, 13])
After layer 4: torch.Size([128, 64, 12, 12])
After avg pooling: torch.Size([128, 64, 6, 6])
After layer 5: torch.Size([128, 64, 5, 5])
After layer 6: torch.Size([128, 64, 4, 4])
After layer 7: torch.Size([128, 22, 3, 3])
After avg pooling: torch.Size([128, 22, 1, 1])
After adaptive pooling: torch.Size([128, 22, 3, 3])
After flattening: torch.Size([128, 198])
After fc1: torch.Size([128, 128])
After fc2: torch.Size([128, 128])
After fc3: torch.Size([128, 64])
After classifier: torch.Size([128, 10])
Epoch 0:   0%|          | 1/391 [00:02<13:40,  0.48it/s, v_num=149, train_loss=2.300, train_acc=0.141]After preliminary layer: torch.Size([128, 16, 30, 30])
After layer 1: torch.Size([128, 64, 29, 29])
After layer 2: torch.Size([128, 64, 28, 28])
After avg po

C:\Users\senti\anaconda3\envs\NNI_NAS_local\Lib\site-packages\pytorch_lightning\trainer\call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...


[2025-02-12 16:12:10] Experiment stopped


## Best architecture

In [26]:
best_arch = experiment_results.export_top_models(formatter = 'instance' , top_k = 1)[0]
best_arch_desc =  experiment_results.export_top_models(formatter = 'dict' , top_k = 1)[0]
best_arch_state_dict = best_arch.state_dict()

In [27]:
best_arch_desc

{'layer_1': 1,
 'layer1_out_channels': 32,
 'layer_2': 1,
 'layer2_out_channels': 16,
 'layer_3': 0,
 'layer3_out_channels': 64,
 'layer_4': 0,
 'layer4_out_channels': 64,
 'layer_5': 0,
 'layer5_out_channels': 16,
 'layer_6': 1,
 'layer6_out_channels': 16,
 'layer_7': 1,
 'feature1': 32,
 'feature2': 32,
 'feature3': 32}

In [17]:
experiment_results.config

NasExperimentConfig(experiment_name='Darts_search', experiment_type='nas', search_space_file=None, search_space='_reserved_', trial_command='', trial_code_directory='.', trial_concurrency=1, trial_gpu_number=None, max_experiment_duration=None, max_trial_number=None, max_trial_duration=None, nni_manager_ip=None, use_annotation=False, debug=False, log_level=None, experiment_working_directory='./DartsCheckpoints', tuner_gpu_indices=None, tuner=_AlgorithmConfig(name='_none_', class_name=None, code_directory=None, class_args={}), assessor=_AlgorithmConfig(name='_none_', class_name=None, code_directory=None, class_args={}), advisor=_AlgorithmConfig(name='_none_', class_name=None, code_directory=None, class_args={}), training_service=LocalConfig(platform='local', trial_command=<dataclasses._MISSING_TYPE object at 0x000002A1DF82C590>, trial_code_directory=<dataclasses._MISSING_TYPE object at 0x000002A1DF82C590>, trial_gpu_number=<dataclasses._MISSING_TYPE object at 0x000002A1DF82C590>, nni_man

# Auto arch code generator

### Load Checkpoint

In [120]:
checkpoint_path = './checkpoints/best-checkpoint-v51.ckpt'

checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

checkpoint_model = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=5,verbose =0)

checkpoint_state_dict = checkpoint_model.state_dict()
pretrained_state_dict = checkpoint['state_dict']

if 'global_step' in checkpoint:
    print("Global Step:", checkpoint['global_step'])

if 'callbacks' in checkpoint and isinstance(checkpoint['callbacks'], dict):
    for key, callback in checkpoint['callbacks'].items():
        if isinstance(callback, dict) and 'best_model_score' in callback:
            print("Best Model Score (Train Accuracy):", callback['best_model_score'].item())

Global Step: 168750
Best Model Score (Train Accuracy): 0.796875


In [121]:
experiment = NasExperiment(CustomDARTSSpace, evaluator, strategy)
experiment.load_checkpoint()
best_arch = experiment_results.export_top_models(formatter = 'instance' , top_k = 1)[0]
best_arch_desc =  experiment_results.export_top_models(formatter = 'dict' , top_k = 1)[0]
best_arch_state_dict = best_arch.state_dict()

[2024-12-20 14:50:44] Config is not provided. Will try to infer.
[2024-12-20 14:50:44] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2024-12-20 14:50:44] WARNING: `training_service` will be ignored for sequential execution engine.


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\senti\\nni-experiments\\2vgea309\\checkpoint\\config.json'

# From-Scratch Model

In [ ]:
class PhotonicArch(torch.nn.Module):
    def __init__(self, drop_path_prob=0.0,arch_dict = None):
        super().__init__()
        self.arch_dict = arch_dict
        self.drop_path_prob = drop_path_prob 
        #________________________________________________________________________________________________________________________
        #Layer 0
        self.layer0_conv = torch.nn.Conv2d(3, 8, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(8)
        self.layer0_relu = torch.nn.ReLU(inplace=False)
        #________________________________________________________________________________________________________________________
        #Layer 1
        if arch_dict['layer_1'] == 0:
            self.layer1_conv=torch.nn.Conv2d(8, arch_dict['layer1_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer1_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer1_bn=torch.nn.BatchNorm2d(arch_dict['layer1_out_channels'], affine=True)
            self.layer1_relu=torch.nn.ReLU(inplace=False)
        else:
            self.layer1_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer1_conv=torch.nn.Conv2d(8, arch_dict['layer1_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer1_bn=torch.nn.BatchNorm2d(arch_dict['layer1_out_channels'], affine=True)
            self.layer1_relu=torch.nn.ReLU(inplace=False)
        #________________________________________________________________________________________________________________________
        #Layer 2 
        if arch_dict['layer_2'] == 0:
            self.layer2_conv=torch.nn.Conv2d(arch_dict['layer1_out_channels'], arch_dict['layer2_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer2_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer2_bn=torch.nn.BatchNorm2d(arch_dict['layer2_out_channels'], affine=True)
            self.layer2_relu=torch.nn.ReLU(inplace=False)
        else:
            self.layer2_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer2_conv=torch.nn.Conv2d(arch_dict['layer1_out_channels'], arch_dict['layer2_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer2_bn=torch.nn.BatchNorm2d(arch_dict['layer2_out_channels'], affine=True)
            self.layer2_relu=torch.nn.ReLU(inplace=False)

        #________________________________________________________________________________________________________________________
        #Layer 3
        if arch_dict['layer_3'] == 0:
            self.layer3_conv=torch.nn.Conv2d(arch_dict['layer2_out_channels'], arch_dict['layer3_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer3_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer3_bn=torch.nn.BatchNorm2d(arch_dict['layer3_out_channels'], affine=True)
            self.layer3_relu=torch.nn.ReLU(inplace=False)
        else:
            self.layer3_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer3_conv=torch.nn.Conv2d(arch_dict['layer2_out_channels'], arch_dict['layer3_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer3_bn=torch.nn.BatchNorm2d(arch_dict['layer3_out_channels'], affine=True)
            self.layer3_relu=torch.nn.ReLU(inplace=False)
        #________________________________________________________________________________________________________________________
        #Layer 4
        if arch_dict['layer_4'] == 0:
            self.layer4_conv=torch.nn.Conv2d(arch_dict['layer3_out_channels'], arch_dict['layer4_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer4_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer4_bn=torch.nn.BatchNorm2d(arch_dict['layer4_out_channels'], affine=True)
            self.layer4_relu=torch.nn.ReLU(inplace=False)
        else:
            self.layer4_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer4_conv=torch.nn.Conv2d(arch_dict['layer3_out_channels'], arch_dict['layer4_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer4_bn=torch.nn.BatchNorm2d(arch_dict['layer4_out_channels'], affine=True)
            self.layer4_relu=torch.nn.ReLU(inplace=False)
        #________________________________________________________________________________________________________________________
        #Layer 5
        if arch_dict['layer_5'] == 0:
            self.layer5_conv=torch.nn.Conv2d(arch_dict['layer4_out_channels'], 22, kernel_size=3, stride=1, padding=1)
            self.layer5_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer5_bn=torch.nn.BatchNorm2d(22, affine=True)
            self.layer5_relu=torch.nn.ReLU(inplace=False)
        else:
            self.layer5_avgpool= torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer5_conv=torch.nn.Conv2d(arch_dict['layer4_out_channels'], 22, kernel_size=3, stride=1, padding=1)
            self.layer5_bn=torch.nn.BatchNorm2d(22, affine=True)
            self.layer5_relu=torch.nn.ReLU(inplace=False)
        #________________________________________________________________________________________________________________________   
        self.pool = torch.nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = torch.nn.Linear(198, 160) 
        self.fc2 = torch.nn.Linear(160, 128) 
        self.fc3 = torch.nn.Linear(128, 96)  
        self.relu = torch.nn.ReLU(inplace=False)
        self.classifier = torch.nn.Linear(96, 10)

    def forward(self, x):
        #________________________________________________________________________________________________________________________
        x = self.layer0_conv(x)
        X = self.layer0_bn(x)
        x = self.layer0_relu(x)
        #________________________________________________________________________________________________________________________
        # Unroll layer1
        x = self.layer1_avgpool(x)
        x = self.layer1_conv(x)
        x = self.layer1_bn(x)
        x = self.layer1_relu(x)
        #print(f'After l1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer2
        x = self.layer2_conv(x)
        x = self.layer2_avgpool(x)
        x = self.layer2_bn(x)
        x = self.layer2_relu(x)
        #print(f'After l2: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer3
        x = self.layer3_conv(x)
        x = self.layer3_avgpool(x)
        x = self.layer3_bn(x)
        x = self.layer3_relu(x)
        #print(f'After l3: {x.shape}')
        #________________________________________________________________________________________________________________________
        # First AvgPool after layer3
        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #print(f'After intermadiate pool 1: {x.shape}')
        #________________________________________________________________________________________________________________________
        # Unroll layer4
        x = self.layer4_avgpool(x)
        x = self.layer4_conv(x)
        x = self.layer4_bn(x)
        x = self.layer4_relu(x)
        #print(f'After l4: {x.shape}')      
        #________________________________________________________________________________________________________________________
        # Unroll layer5
        x = self.layer5_avgpool(x)
        x = self.layer5_conv(x)
        x = self.layer5_bn(x)
        x = self.layer5_relu(x)
        #print(f'After l5: {x.shape}')     
        #________________________________________________________________________________________________________________________
        # second AvgPool after layer5
        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        #print(f'After intermediate pool 2 {x.shape}')
        #________________________________________________________________________________________________________________________
        x =  self.pool(x)
        #print(f'After adaptive: {x.shape}')
        #________________________________________________________________________________________________________________________
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x= self.relu(x)
        x = self.fc2(x)
        x= self.relu(x)
        x = self.fc3(x)
        x= self.relu(x)
        
        x = self.classifier(x)
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob

In [23]:
scratch_model = PhotonicArch(arch_dict = best_arch_desc)

In [24]:
scratch_model

PhotonicArch(
  (layer0_conv): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), bias=False)
  (layer0_bn): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer0_relu): ReLU()
  (layer1_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer1_conv): Conv2d(8, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer1_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1_relu): ReLU()
  (layer2_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer2_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer2_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer2_relu): ReLU()
  (layer3_conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer3_avgpool): AvgPool2d(kernel_size=3, stride=1, padding=0)
  (layer3_bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer3_re

### Rename dictionary keys

In [25]:
# Preliminary layer
best_arch_state_dict['layer0_conv.weight'] = best_arch_state_dict.pop('preliminary_layer.weight')
best_arch_state_dict['layer0_bn.weight'] = best_arch_state_dict.pop('layer0_bn.weight')
best_arch_state_dict['layer0_bn.bias'] = best_arch_state_dict.pop('layer0_bn.bias')
best_arch_state_dict['layer0_bn.running_mean'] = best_arch_state_dict.pop('layer0_bn.running_mean')
best_arch_state_dict['layer0_bn.running_var'] = best_arch_state_dict.pop('layer0_bn.running_var')
best_arch_state_dict['layer0_bn.num_batches_tracked'] = best_arch_state_dict.pop('layer0_bn.num_batches_tracked')
# Layer 1
if 'layers.0.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer1_conv.weight'] = best_arch_state_dict.pop('layers.0.0.weight')
    best_arch_state_dict['layer1_conv.bias'] = best_arch_state_dict.pop('layers.0.0.bias')
else:
    best_arch_state_dict['layer1_conv.weight'] = best_arch_state_dict.pop('layers.0.1.weight')
    best_arch_state_dict['layer1_conv.bias'] = best_arch_state_dict.pop('layers.0.1.bias')   
best_arch_state_dict['layer1_bn.weight'] = best_arch_state_dict.pop('layers.0.2.weight')
best_arch_state_dict['layer1_bn.bias'] = best_arch_state_dict.pop('layers.0.2.bias')
best_arch_state_dict['layer1_bn.running_mean'] = best_arch_state_dict.pop('layers.0.2.running_mean')
best_arch_state_dict['layer1_bn.running_var'] = best_arch_state_dict.pop('layers.0.2.running_var')
best_arch_state_dict['layer1_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.0.2.num_batches_tracked')

#Layer 2
if 'layers.1.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer2_conv.weight'] = best_arch_state_dict.pop('layers.1.0.weight')
    best_arch_state_dict['layer2_conv.bias'] = best_arch_state_dict.pop('layers.1.0.bias')
else:
    best_arch_state_dict['layer2_conv.weight'] = best_arch_state_dict.pop('layers.1.1.weight')
    best_arch_state_dict['layer2_conv.bias'] = best_arch_state_dict.pop('layers.1.1.bias')   
best_arch_state_dict['layer2_bn.weight'] = best_arch_state_dict.pop('layers.1.2.weight')
best_arch_state_dict['layer2_bn.bias'] = best_arch_state_dict.pop('layers.1.2.bias')
best_arch_state_dict['layer2_bn.running_mean'] = best_arch_state_dict.pop('layers.1.2.running_mean')
best_arch_state_dict['layer2_bn.running_var'] = best_arch_state_dict.pop('layers.1.2.running_var')
best_arch_state_dict['layer2_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.1.2.num_batches_tracked')

#Layer 3
if 'layers.2.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer3_conv.weight'] = best_arch_state_dict.pop('layers.2.0.weight')
    best_arch_state_dict['layer3_conv.bias'] = best_arch_state_dict.pop('layers.2.0.bias')
else:
    best_arch_state_dict['layer3_conv.weight'] = best_arch_state_dict.pop('layers.2.1.weight')
    best_arch_state_dict['layer3_conv.bias'] = best_arch_state_dict.pop('layers.2.1.bias')   
best_arch_state_dict['layer3_bn.weight'] = best_arch_state_dict.pop('layers.2.2.weight')
best_arch_state_dict['layer3_bn.bias'] = best_arch_state_dict.pop('layers.2.2.bias')
best_arch_state_dict['layer3_bn.running_mean'] = best_arch_state_dict.pop('layers.2.2.running_mean')
best_arch_state_dict['layer3_bn.running_var'] = best_arch_state_dict.pop('layers.2.2.running_var')
best_arch_state_dict['layer3_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.2.2.num_batches_tracked')

#Layer 4
if 'layers.3.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer4_conv.weight'] = best_arch_state_dict.pop('layers.3.0.weight')
    best_arch_state_dict['layer4_conv.bias'] = best_arch_state_dict.pop('layers.3.0.bias')
else:
    best_arch_state_dict['layer4_conv.weight'] = best_arch_state_dict.pop('layers.3.1.weight')
    best_arch_state_dict['layer4_conv.bias'] = best_arch_state_dict.pop('layers.3.1.bias')   
best_arch_state_dict['layer4_bn.weight'] = best_arch_state_dict.pop('layers.3.2.weight')
best_arch_state_dict['layer4_bn.bias'] = best_arch_state_dict.pop('layers.3.2.bias')
best_arch_state_dict['layer4_bn.running_mean'] = best_arch_state_dict.pop('layers.3.2.running_mean')
best_arch_state_dict['layer4_bn.running_var'] = best_arch_state_dict.pop('layers.3.2.running_var')
best_arch_state_dict['layer4_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.3.2.num_batches_tracked')

#Layer 5
if 'layers.4.0.weight' in best_arch_state_dict:
    best_arch_state_dict['layer5_conv.weight'] = best_arch_state_dict.pop('layers.4.0.weight')
    best_arch_state_dict['layer5_conv.bias'] = best_arch_state_dict.pop('layers.4.0.bias')
else:
    best_arch_state_dict['layer5_conv.weight'] = best_arch_state_dict.pop('layers.4.1.weight')
    best_arch_state_dict['layer5_conv.bias'] = best_arch_state_dict.pop('layers.4.1.bias')   
best_arch_state_dict['layer5_bn.weight'] = best_arch_state_dict.pop('layers.4.2.weight')
best_arch_state_dict['layer5_bn.bias'] = best_arch_state_dict.pop('layers.4.2.bias')
best_arch_state_dict['layer5_bn.running_mean'] = best_arch_state_dict.pop('layers.4.2.running_mean')
best_arch_state_dict['layer5_bn.running_var'] = best_arch_state_dict.pop('layers.4.2.running_var')
best_arch_state_dict['layer5_bn.num_batches_tracked'] = best_arch_state_dict.pop('layers.4.2.num_batches_tracked')



KeyError: 'preliminary_layer.weight'

## Load weights

In [26]:
scratch_model.load_state_dict(best_arch_state_dict)

<All keys matched successfully>

In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def evaluate_model(model, dataloader, criterion, device):

    model.eval()  # Set the model to evaluation mode
    total_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():  # Disable gradient computation for evaluation
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Compute predictions and accuracy
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total_samples

    return avg_loss, accuracy


In [28]:

from torchvision.transforms import ToTensor
from pytorch_lightning import Trainer
scratch_model.train()
module = DartsClassificationModule(1e-5, 0., 0., max_epochs)
module.set_model(scratch_model)
# Train
trainer = Trainer(max_epochs=module.max_epochs,precision=16,gradient_clip_val=0.1)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(module, train_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params
-----------------------------------------------
0 | criterion | CrossEntropyLoss | 0     
1 | metrics   | ModuleDict       | 0     
2 | _model    | PhotonicArch     | 194 K 
-----------------------------------------------
194 K     Trainable params
0         Non-trainable params
194 K     Total params
0.779     Total estimated model params size (MB)


Epoch 172:  96%|█████████▋| 302/313 [00:07<00:00, 38.05it/s, v_num=117, train_loss=0.811, train_acc=0.703]

In [ ]:
criterion = nn.CrossEntropyLoss()
scratch_model.to(device)

avg_loss, accuracy = evaluate_model(scratch_model, valid_loader, criterion, device)
print(f"Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4%}")


In [ ]:
for inputs, labels in valid_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = scratch_model(inputs)
    print("Predicted:", outputs.argmax(dim=1))
    print("True:", labels)
    break
